# wp1pt4pt1i — site-specific 2-spring SDOF parameterisation & optimisation setup

For each case-study site this notebook turns the MDOF cyclic-pushover (CPO) responses into a
calibrated **2-spring equivalent SDOF** (following `wp1pt4pt1e`):

* **Spring 1 — `Steel02`**: the soft-storey / frame mechanism, backbone fitted from the
  **mechanism** CPO (`site_{ii}/mdof_mechanism`).
* **Spring 2 — `Hysteretic`**: the brace contribution, backbone fitted from the
  **full-frame minus mechanism** CPO.

It (1) saves the spring backbone properties per site to
`data_processed/10_site_specific_sdofs/site_{ii}/3s_cbf_dc2_site{ii}_sdof_props_specific.json`,
and (2) sets up the differential-evolution optimisation of the hysteretic parameters (per-site
optimisation inputs under `site_{ii}/optimisation/` + one batch launcher script).

This is the **direct-extraction** path — backbones from the actual CPO data, hysteretic shape from
optimisation. The regression/equation-prediction path and any statistics are intentionally excluded.

**Prerequisites** (on the `D:` drive): the intact CPO (`site_{ii}/mdof/cyclic_pushover/po_curve.csv`,
already run) and the **mechanism CPO** (`site_{ii}/mdof_mechanism/cyclic_pushover/po_curve.csv`, run via
`site_mdof_mechanism_cpo.py`). `participation_factors.json` (from `wp1pt4pt1f`) supplies `Gamma`, `m_star`.


In [ ]:
%load_ext autoreload
%autoreload 2

## Setup & parameters

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from fitpo import fit_envelope, fit_piecewise_backbone, fit_tetralinear_backbone

from phd_project.scripts.standardise_responses import standardise_responses
from phd_project.scripts.templates.copy_templates_to_folders import configure_optimisation_batch_run
from phd_project.config import config

cfg = config.load_config()

In [ ]:
# --- locations ---
ROOT = Path(r"D:\04_site_influence_investigation")        # site analysis folders (mdof, mdof_mechanism)
OUT_ROOT = Path(cfg["proc_data"]["site_specific_sdofs"])  # data_processed/10_site_specific_sdofs

# --- structure ---
N_STOREYS, DUCTILITY_CLASS = 3, 2
def site_tag(ii: int) -> str:
    return f"{N_STOREYS}s_cbf_dc{DUCTILITY_CLASS}_site{ii}"

# --- brace tetralinear backbone (match wp1pt4pt1e) ---
RESIDUAL_FORCE_FRACTION, COLLAPSE_RESIDUAL = 0.1, True

# --- differential-evolution optimisation controls (match wp1pt4pt1e) ---
POPSIZE, CORES, SEED, TOL = 15, 60, 1, 1e-8        # CORES: adjust to the machine
MUTATION, RECOMBINATION, DU_MAX = (0.5, 1), 0.7, 2.0

# hysteretic optimisation parameter bounds:
#   a1, a2, R0, cR1, cR2 (Steel02)  +  pinchX, pinchY, damage1, damage2, beta (Hysteretic)
HYST_PARAM_BOUNDS = [(0, 0.5), (1.0, 6.0), (10, 25), (0.5, 0.95), (0.01, 1.0),
                     (0.01, 1.0), (0.01, 1.0), (0, 1.0), (0, 1.0), (0, 1.0)]

## Discover ready sites

A site is *ready* when both its intact (`mdof`) and mechanism (`mdof_mechanism`) cyclic-pushover
curves exist. `Gamma` and `m_star` come from `participation_factors.json` (produced by `wp1pt4pt1f`).

In [ ]:
with open(OUT_ROOT / "participation_factors.json") as f:
    participation = {int(k): v for k, v in json.load(f).items()}


# load a cyclic-pushover F-D curve ([disp, load]) from <folder>/cyclic_pushover/po_curve.csv
def _load_cpo_curve(folder: Path) -> np.ndarray:
    return np.loadtxt(Path(folder) / "cyclic_pushover" / "po_curve.csv", delimiter=",")


sites = []
for ii in sorted(participation):
    full = ROOT / f"site_{ii}" / "mdof" / "cyclic_pushover" / "po_curve.csv"
    mech = ROOT / f"site_{ii}" / "mdof_mechanism" / "cyclic_pushover" / "po_curve.csv"
    if full.exists() and mech.exists():
        sites.append(ii)
    else:
        missing = [n for n, p in [("mdof", full), ("mdof_mechanism", mech)] if not p.exists()]
        print(f"skip site {ii}: missing CPO -> {', '.join(missing)}")

print(f"{len(sites)} of {len(participation)} sites ready")

## Parameterise a site

Faithful to `wp1pt4pt1e`:

* **Steel02 (soft-storey) backbone** — envelope of the mechanism CPO, fitted bilinear with a negative
  post-yield slope; `Fy`, `du` converted to SDOF coordinates by the full-system `Gamma` (`E0`, `b` are
  ratio quantities and stay invariant).
* **Hysteretic (brace) backbone** — the brace-only cyclic curve (full minus mechanism, on a common
  displacement grid), enveloped and fitted tetralinear (collapse-to-zero); flattened into the
  positive-then-negative `Hysteretic` point list and divided by `Gamma`.

The saved dict has **exactly** `steel02_bb_params`, `hysteretic_bb_params`, `mass` so it doubles as the
optimiser's `sdof_params` input (consumed via `initialise_steel02_hysteretic_model(**sdof_const)`).

In [ ]:
def parameterise_site(ii: int):
    gamma = participation[ii]["Gamma"]
    m_star = participation[ii]["m_star"]

    cpoc = _load_cpo_curve(ROOT / f"site_{ii}" / "mdof")             # full frame
    cpoc_ss = _load_cpo_curve(ROOT / f"site_{ii}" / "mdof_mechanism")  # soft-storey mechanism

    # --- Spring 1 (Steel02): soft-storey backbone from the mechanism CPO ---
    env_ss, _ = fit_envelope(cpoc_ss)
    bb = fit_piecewise_backbone(env_ss, mode="bilinear_incl_neg_stiffness")
    ke = bb[1, 1] / bb[1, 0]
    kh = (bb[2, 1] - bb[1, 1]) / (bb[2, 0] - bb[1, 0])
    Fy, b, du = bb[1, 1], kh / ke, abs(bb[1, 1] / kh)
    steel02_bb_params = [Fy / gamma, ke, b, du / gamma]   # SDOF coords (Fy, du /gamma; E0, b invariant)

    # --- brace-only cyclic curve = full - soft-storey, on a common displacement grid ---
    tot, ss, _ = standardise_responses(cpoc, cpoc_ss)
    cpoc_br = np.column_stack([tot[:, 0], tot[:, 1] - ss[:, 1]])

    # --- Spring 2 (Hysteretic): brace backbone (tetralinear, collapse-to-zero) ---
    env_br, _ = fit_envelope(cpoc_br)
    bb_br = fit_tetralinear_backbone(
        env_br, residual_force_fraction=RESIDUAL_FORCE_FRACTION, collapse_residual=COLLAPSE_RESIDUAL)
    sl = slice(1, -1) if COLLAPSE_RESIDUAL else slice(1, None)
    hyst_bb = np.concatenate([bb_br[sl, [1, 0]], -bb_br[sl, [1, 0]]]).flatten()
    hysteretic_bb_params = (hyst_bb / gamma).tolist()

    props = {
        "steel02_bb_params": steel02_bb_params,
        "hysteretic_bb_params": hysteretic_bb_params,
        "mass": m_star,
    }
    return props, cpoc, cpoc_ss, gamma

## Build the props JSON + optimisation inputs

Per site: write the requested `..._sdof_props_specific.json` into `site_{ii}/`, and the optimisation
inputs (full-frame target hysteresis, mechanism target-displacement path, parameter bounds) into
`site_{ii}/optimisation/`. Everything is in equivalent-SDOF coordinates (`/Gamma`).

In [ ]:
configs = []
sdof_props = {}

for ii in tqdm(sites, desc="Parameterising sites"):
    tag = site_tag(ii)
    site_dir = OUT_ROOT / f"site_{ii}"
    opt_dir = site_dir / "optimisation"
    site_dir.mkdir(parents=True, exist_ok=True)
    opt_dir.mkdir(parents=True, exist_ok=True)

    props, cpoc, cpoc_ss, gamma = parameterise_site(ii)
    sdof_props[ii] = props

    # (1) the requested SDOF properties file
    props_path = site_dir / f"{tag}_sdof_props_specific.json"
    with open(props_path, "w") as f:
        json.dump(props, f, indent=4)

    # (2) optimisation inputs (equivalent-SDOF coordinates)
    test_data_path = opt_dir / f"{tag}_eq_sdof_cpo.csv"          # full-frame target hysteresis
    target_disps_path = opt_dir / f"{tag}_target_displacements.csv"  # mechanism displacement path
    bounds_path = opt_dir / f"{tag}_steel02_hysteretic_parameter_bounds.csv"
    np.savetxt(test_data_path, cpoc / gamma, delimiter=",")
    np.savetxt(target_disps_path, cpoc_ss[:, 0] / gamma, delimiter=",")
    np.savetxt(bounds_path, HYST_PARAM_BOUNDS, delimiter=",")

    configs.append({
        "results_folder": opt_dir,
        "result_name": "_steel02_and_hysteretic_optimisation.pickle",
        "building_tag": tag,
        "sdof_params": props_path,
        "test_data": test_data_path,
        "displacements": target_disps_path,
        "dU_max": DU_MAX,
        "param_bounds": bounds_path,
        "initialise_model_func": "steel02_and_hysteretic",
        "popsize": POPSIZE,
        "cores": CORES,
        "seed": SEED,
        "tol": TOL,
        "mutation": MUTATION,
        "recombination": RECOMBINATION,
    })

print(f"wrote SDOF props + optimisation inputs for {len(configs)} sites -> {OUT_ROOT}")

### Diagnostic plot (spot-check a few fits)

Mechanism CPO with the Steel02 (soft-storey) backbone, and the brace-only CPO with the tetralinear
Hysteretic backbone, for the first few ready sites.

In [ ]:
n_show = min(6, len(sites))
if n_show:
    fig, axs = plt.subplots(n_show, 2, figsize=(9, 2.6 * n_show), squeeze=False)
    for row, ii in enumerate(sites[:n_show]):
        gamma = participation[ii]["Gamma"]
        cpoc = _load_cpo_curve(ROOT / f"site_{ii}" / "mdof")
        cpoc_ss = _load_cpo_curve(ROOT / f"site_{ii}" / "mdof_mechanism")

        env_ss, _ = fit_envelope(cpoc_ss)
        bb = fit_piecewise_backbone(env_ss, mode="bilinear_incl_neg_stiffness")

        tot, ss, _ = standardise_responses(cpoc, cpoc_ss)
        cpoc_br = np.column_stack([tot[:, 0], tot[:, 1] - ss[:, 1]])
        env_br, _ = fit_envelope(cpoc_br)
        bb_br = fit_tetralinear_backbone(
            env_br, residual_force_fraction=RESIDUAL_FORCE_FRACTION, collapse_residual=COLLAPSE_RESIDUAL)

        ax = axs[row, 0]
        ax.plot(cpoc_ss[:, 0], cpoc_ss[:, 1], color="0.8", ls="-.", label="mechanism CPO")
        ax.plot(env_ss[:, 0], env_ss[:, 1], color="r", ls="--", label="envelope")
        ax.plot(bb[:, 0], bb[:, 1], color="b", ls="-.", label="Steel02 BB")
        ax.set_title(f"site {ii} — soft storey")
        ax.grid(ls="-.", color="0.85")

        ax = axs[row, 1]
        ax.plot(cpoc_br[:, 0], cpoc_br[:, 1], color="0.8", ls="-.", label="brace CPO")
        ax.plot(env_br[:, 0], env_br[:, 1], color="r", ls="--", label="envelope")
        ax.plot(bb_br[:, 0], bb_br[:, 1], color="g", ls="-.", label="Hysteretic BB")
        ax.set_title(f"site {ii} — braces")
        ax.grid(ls="-.", color="0.85")

    axs[0, 0].legend(fontsize=8)
    axs[0, 1].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

## Write the optimisation batch launcher

One batch script listing every ready site's `steel02_and_hysteretic` optimisation job. Distinct name so
it does not clobber `wp1pt4pt1e`'s `optimisation_batch_run_3s_steel02_and_hysteretic.py`. Launch it to run
the differential-evolution calibration; each site's result pickle lands in its `optimisation/` folder.

In [ ]:
src = cfg["templates"]["group_sdof_optimisation"]
dst = Path(cfg["scripts"]["wp1pt4pt1_batch_run"]) / "optimisation_batch_run_site_steel02_and_hysteretic.py"
configure_optimisation_batch_run(src, dst, configs)
print(f"wrote {dst}\n  with {len(configs)} site optimisation jobs")